# 2 · Ejercicio 1: Curry-Howard (25 min)

**Idea:** demostrar una fórmula = escribir una función de Scala con ese tipo. Si compila sin `???`, está demostrada.

### Preámbulo
Ejecuta esta celda antes que las demás: son las mismas definiciones que trae el examen.

In [ ]:
import $ivy.`org.scalatest::scalatest:3.2.16`
import org.scalatest.{Filter => _, _}, flatspec._, matchers._

type Not[P] = P => Nothing
type <=>[P, Q] = (P => Q, Q => P)

trait ExcludedMiddle:
    def apply[P]: Either[P, Not[P]]

trait DoubleNegation:
    def apply[P]: Not[Not[P]] => P

trait Isomorphic[A, B]:

    def from(a: A): B

    def to(b: B): A

    // equality

    def equalA(a1: A, a2: A): Boolean =
        a1 == a2

    def equalB(b1: B, b2: B): Boolean =
        b1 == b2

    // Bijection laws

    def law1(a: A): Boolean =
        equalA(to(from(a)), a)

    def law2(b: B): Boolean =
        equalB(from(to(b)), b)

// type Tree[A] = 1 + Tree[A] * A * Tree[A]

enum Tree[A]:
    case Empty()
    case Node(left: Tree[A], root: A, right: Tree[A])

object Tree:

    def void[A]: Tree[A] =
        Empty()

    def leaf[A](a: A): Node[A] =
        Node(Empty(), a, Empty())

    def right[A](a: A, tree: Tree[A]): Node[A] =
        Node(Empty(), a, tree)

    def left[A](tree: Tree[A], a: A): Node[A] =
        Node(tree, a, Empty())

    def node[A](left: Tree[A], a: A, right: Tree[A]): Node[A] =
        Node(left, a, right)

import Tree._

def foldTree[A, B](tree: Tree[A])(empty: B)(node: (B, A, B) => B): B =
    tree match
        case Empty() => empty
        case Node(left, a, right) =>
            node(foldTree(left)(empty)(node), a, foldTree(right)(empty)(node))

## Paso 1: traducir

| Lógica | Scala |
|---|---|
| p → q | `P => Q` |
| p ∧ q | `(P, Q)` |
| p ∨ q | `Either[P, Q]` |
| ¬p | `Not[P]` (= `P => Nothing`) |
| ⊥ | `Nothing` |

**Precedencia:** ¬ antes que ∧, antes que ∨, antes que →. La → agrupa a la **derecha**: `a → b → c` = `a → (b → c)`.

- `p ∧ q → r` es `((P, Q)) => R` (¡doble paréntesis: recibe **un** par!).
- Un argumento `{premisa1, premisa2} ⊢ conclusión` es una función con parámetros `(p1: ..., p2: ...)` que devuelve la conclusión.

## Paso 2: construir con "meta" y "recursos"

La **meta** es el tipo que te falta construir; los **recursos**, las variables que tienes.

| Si la meta es… | escribe |
|---|---|
| `P => Q` o `Not[P]` | `p => ...` (ganas `p`) |
| `(P, Q)` | `(..., ...)` |
| `Either[P, Q]` | `Left(...)` o `Right(...)` |
| `Nothing` | aplica una negación que tengas: `np(p)` |

| Si tienes… | puedes |
|---|---|
| `f: P => Q` y `p: P` | `f(p): Q` |
| `t: (P, Q)` | `t._1`, `t._2` o `(p, q) =>` |
| `e: Either[P, Q]` | `e match { case Left(p) => ...; case Right(q) => ... }` |
| algo de tipo `Nothing` | ¡úsalo como **cualquier** tipo! |

## Ejemplo 1 · ⊢ (p → q) → (q → r) → p → r

In [ ]:
def proof[P, Q, R]: (P => Q) => (Q => R) => P => R =
    f => g => p => g(f(p))

## Ejemplo 2 · ⊢ ¬p ∨ ¬q → ¬(p ∧ q)

Tenemos un `Either` → `match`. La meta de cada rama es `Not[(P, Q)]` → lambda que recibe el par.

In [ ]:
def proof[P, Q]: Either[Not[P], Not[Q]] => Not[(P, Q)] =
    case Left(np)  => (p, _) => np(p)
    case Right(nq) => (_, q) => nq(q)

## Ejemplo 3 · {p → q ∨ r, ¬r} ⊢ p → q (con premisas)

In [ ]:
def proof[P, Q, R](p1: P => Either[Q, R], p2: Not[R]): P => Q =
    p => p1(p) match
        case Left(q)  => q
        case Right(r) => p2(r)      // Nothing: vale como Q

## Apartado b): lógica clásica

Te dejan usar una ayuda **sobre una proposición concreta** (la que diga el enunciado):

- **Tercio excluso** (`lem: ExcludedMiddle`): `lem[X]` te da `Either[X, Not[X]]` → haz `match`.
- **Doble negación** (`dn: DoubleNegation`): si la meta es `X`, escribe `dn[X](nx => ...)`; dentro la meta es `Nothing` y tienes `nx: Not[X]`.

## Ejemplo 4 · ⊢ (a → b) → ¬a ∨ b, con tercio excluso en a

In [ ]:
def proof[A, B](lem: ExcludedMiddle): (A => B) => Either[Not[A], B] =
    f => lem[A] match
        case Left(a)   => Right(f(a))
        case Right(na) => Left(na)

## Ejemplo 5 · ⊢ ((p → q) → p) → p, con doble negación en p

In [ ]:
def proof[P, Q](dn: DoubleNegation): ((P => Q) => P) => P =
    f => dn[P](np => np(f(np)))     // np: P => Nothing sirve como P => Q

## Mini-ejercicios (5 min)

Rellena los `???` y comprueba que compila.

**1.** ⊢ p ∧ q → q ∧ p

In [ ]:
def ej1[P, Q]: ((P, Q)) => (Q, P) =
    ???

<details>
<summary><b>Ver solución</b></summary>

```scala
def ej1[P, Q]: ((P, Q)) => (Q, P) =
    (p, q) => (q, p)
```

</details>

**2.** ⊢ (p → q) → ¬q → ¬p

In [ ]:
def ej2[P, Q]: (P => Q) => Not[Q] => Not[P] =
    ???

<details>
<summary><b>Ver solución</b></summary>

```scala
def ej2[P, Q]: (P => Q) => Not[Q] => Not[P] =
    f => nq => p => nq(f(p))
```

</details>

**3.** ⊢ ¬¬p → p, con tercio excluso en p

In [ ]:
def ej3[P](lem: ExcludedMiddle): Not[Not[P]] => P =
    ???

<details>
<summary><b>Ver solución</b></summary>

```scala
def ej3[P](lem: ExcludedMiddle): Not[Not[P]] => P =
    nnp => lem[P] match
        case Left(p)   => p
        case Right(np) => nnp(np)
```

</details>

**Siguiente:** `03-ejercicio2-isomorfismos.ipynb`